# Hyperparameter Optimization: Grid Search vs Bayesian

**Dataset:** California Housing

**Objective:** Compare two popular methods for finding the best model settings: exhaustive **Grid Search** and efficient **Bayesian Optimization**.

**Key Concepts:**

- **Grid Search:** Brute-force checking of every combination in a predefined grid.
- **Bayesian Optimization:** Uses a probabilistic model to "guess" the next best set of parameters to try, based on previous results. Usually much faster for large search spaces.

---


### Step 1: Setup & Data Loading


In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space import Integer, Real

# Load & Scale
california = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(california.data, california.target, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Setup complete.")

Setup complete.


### Step 2: Grid Search (Exhaustive)

We use `GridSearchCV` to tune a Support Vector Regressor (SVR). To save time in this lab, we fit on a subset of 2,000 samples.


In [2]:
param_grid = {
    'C': [0.1, 1.0, 5.0],
    'epsilon': [0.1, 0.2]
}

grid_search = GridSearchCV(SVR(kernel='rbf'), param_grid, cv=3, scoring='r2', n_jobs=-1)

# Fit on subset for speed
grid_search.fit(X_train_scaled[:2000], y_train[:2000])

print(f"Best SVR Parameters (Grid Search): {grid_search.best_params_}")

Best SVR Parameters (Grid Search): {'C': 5.0, 'epsilon': 0.2}


### Step 3: Bayesian Optimization (Efficient)

We use `BayesSearchCV` from `scikit-optimize` to tune a Random Forest Regressor.


In [3]:
search_space = {
    'n_estimators': Integer(50, 150),
    'max_depth': Integer(5, 20),
    'min_samples_split': Integer(2, 10)
}

bayes_search = BayesSearchCV(
    RandomForestRegressor(random_state=42),
    search_space,
    n_iter=10, # Number of parameter settings that are sampled
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

bayes_search.fit(X_train_scaled[:5000], y_train[:5000]) # Subset for speed

print(f"Best RF Parameters (Bayesian): {dict(bayes_search.best_params_)}")

Best RF Parameters (Bayesian): {'max_depth': 17, 'min_samples_split': 3, 'n_estimators': 110}


### Step 4: Compare & Save

Bayesian optimization is generally preferred for models with many hyperparameters because it learns from its own history to avoid wasting time on poor settings.
